# 04 — Train a CLS-preserving residual graph model

This notebook is a thin Colab entry point. Reusable code lives in `src/cross_image_glot`.

In [1]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")

if "<YOUR_GITHUB_USERNAME>" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository before running this notebook.")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


Mounted at /content/drive
Cloning into '/content/CrossImagePatchGraph_repo'...
remote: Enumerating objects: 71, done.
remote: Counting objects: 100% (71/71), done.
remote: Compressing objects: 100% (49/49), done.
remote: Total 71 (delta 31), reused 52 (delta 18), pack-reused 0 (from 0)
Receiving objects: 100% (71/71), 73.63 KiB | 1.67 MiB/s, done.
Resolving deltas: 100% (31/31), done.
/content/CrossImagePatchGraph_repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 69.7 MB/s eta 0:00:00
  Building editable for cross-image-glot (pyproject.toml) ... done


In [2]:
import json
import torch

from cross_image_glot.config import DEFAULT_PATHS

paths = DEFAULT_PATHS
paths.ensure_directories()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local runtime root:", paths.local_root)

Device: cuda
Drive root: /content/drive/MyDrive/CrossImagePatchGraph
Local runtime root: /content/CrossImagePatchGraph_runtime


The final score is `CLS logits + α × graph logits`. `α` starts at zero, so the initial model exactly matches the frozen CLS baseline instead of destroying it.

In [4]:
from cross_image_glot.models import CrossImageGraphMatcher, BaselinePreservingResidualMatcher
from cross_image_glot.training import (
    evaluate_residual_dataset, load_training_checkpoint, make_checkpoint,
    save_checkpoint_atomic, save_history, train_residual_epoch,
)

from cross_image_glot.storage import restore_feature_splits
from cross_image_glot.data import MiniImageNetFeatureDataset, FewShotFeatureEpisodeDataset
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder
drive.mount("/content/drive",force_remount=True)
restore_feature_splits(["train", "val"], paths.drive_feature_dir, paths.local_feature_dir)
train_features = MiniImageNetFeatureDataset(paths.local_feature_dir, "train", max_cached_shards=6)
val_features = MiniImageNetFeatureDataset(paths.local_feature_dir, "val", max_cached_shards=6)

config = json.loads(Path("configs/residual_5shot.json").read_text())
train_episodes = FewShotFeatureEpisodeDataset(
    train_features, config["n_way"], config["k_shot"], config["train_queries_per_class"],
    1000, config["train_seed"], True,
)
val_episodes = FewShotFeatureEpisodeDataset(
    val_features, config["n_way"], config["k_shot"], config["eval_queries_per_class"],
    600, config["val_seed"], False,
)
graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(train_features.metadata["grid_size"]), top_k=config["top_k"],
    min_similarity=None, graph_dtype=torch.float32, similarity_device=device,
)
graph_matcher = CrossImageGraphMatcher(
    input_dim=config["input_dim"], hidden_dim=config["hidden_dim"], num_layers=config["num_layers"],
    dropout=config["dropout"], temperature=config["graph_temperature"], learnable_temperature=False,
)
model = BaselinePreservingResidualMatcher(graph_matcher, config["initial_residual_scale"]).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])

Mounted at /content/drive


In [5]:
checkpoint_dir = paths.drive_checkpoint_dir / config["experiment_name"]
result_dir = paths.drive_results_dir / config["experiment_name"]
checkpoint_dir.mkdir(parents=True, exist_ok=True)
result_dir.mkdir(parents=True, exist_ok=True)
latest, best = checkpoint_dir / "latest.pt", checkpoint_dir / "best.pt"
history, start_epoch = [], 0
best_accuracy, without_improvement = float("-inf"), 0
RESUME = True
if RESUME and latest.exists():
    state = load_training_checkpoint(latest, model, optimizer, device)
    start_epoch = state["epoch"] + 1
    best_accuracy = state["best_validation_accuracy"]
    without_improvement = state["epochs_without_improvement"]
    history = state.get("history", [])

In [ ]:
for epoch in range(start_epoch, config["num_epochs"]):
    print(f"\nEpoch {epoch + 1}/{config['num_epochs']}")
    train_metrics = train_residual_epoch(
        model, optimizer, graph_builder, train_episodes, device, epoch,
        config["train_episodes_per_epoch"], config["graph_microbatch_size"],
        config["cls_temperature"], log_interval=10,
    )
    val_metrics = evaluate_residual_dataset(
        model, graph_builder, val_episodes, device,
        config["validation_episodes_per_epoch"], config["graph_microbatch_size"],
        config["cls_temperature"], log_interval=5,
    )
    record = {
        "epoch": epoch,
        "train_loss": train_metrics.loss,
        "train_accuracy": train_metrics.accuracy,
        "validation_loss": val_metrics.loss,
        "validation_accuracy": val_metrics.accuracy,
        "residual_scale": float(model.residual_scale.detach().cpu()),
    }
    history.append(record)
    improved = val_metrics.accuracy > best_accuracy
    if improved:
        best_accuracy, without_improvement = val_metrics.accuracy, 0
    else:
        without_improvement += 1
    state = make_checkpoint(model, optimizer, epoch, best_accuracy, without_improvement, history, config)
    save_checkpoint_atomic(state, latest)
    if improved:
        save_checkpoint_atomic(state, best)
    save_history(history, result_dir)
    print(record, "best=", best_accuracy)
    if without_improvement >= config["early_stopping_patience"]:
        print("Early stopping.")
        break

In [6]:
state = torch.load(best, map_location="cpu", weights_only=False)
model.load_state_dict(state["model_state_dict"])
model.to(device)
final_metrics = evaluate_residual_dataset(
    model, graph_builder, val_episodes, device,
    config["final_validation_episodes"], config["graph_microbatch_size"],
    config["cls_temperature"], log_interval=10,
)
from cross_image_glot.storage import atomic_json_save
atomic_json_save({**final_metrics.to_dict(), "residual_scale": float(model.residual_scale.detach().cpu())}, result_dir / "validation_metrics.json")
print(final_metrics)
print("Residual scale:", model.residual_scale.item())

KeyboardInterrupt: 